# 06 — Testing & Evaluation

**Inputs needed:** All Phase-2/3/4/5 artefacts and trained checkpoints.
**Outputs produced:** `results/evaluation_metrics.json`, ROC + ablation plots in `Plots/`, sanity-check tables.
**Runtime:** A few minutes for the full evaluation pass on CPU/GPU.


Phase 6 + design-doc testing strategy:

- **Pipeline sanity checks** — phase filtering, segmentation outputs, Z-score distributions.
- **Patient-level cross-validation** for the radiomics baseline.
- **Ablation comparison** over `results/evaluation_metrics.json`.
- **Interpretability sanity check** — confirm attention maps live inside the liver mask.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import nibabel as nib
import numpy as np
import pandas as pd

from src.utils.config import load_config

cfg = load_config(ROOT / "configs" / "default.yaml")
processed_dir = Path(cfg["paths"]["processed_dir"])
attention_dir = Path(cfg["paths"]["attention_dir"])
results_dir = Path(cfg["paths"]["results_dir"])
labels_csv = Path(cfg["paths"]["labels_csv"])

## 1. Pipeline sanity checks

In [ ]:
labels = pd.read_csv(labels_csv)
expected = {"before.nii.gz", "before_liver.nii.gz", "before_cropped.nii.gz", "crop_metadata.json"}
rows = []
for _, row in labels.iterrows():
    pid = str(row["patient_id"])
    pdir = processed_dir / pid
    present = {p.name for p in pdir.iterdir()} if pdir.exists() else set()
    missing = expected - present
    rows.append({"patient_id": pid, "label": int(row["label"]), "missing": sorted(missing)})
sanity = pd.DataFrame(rows)
sanity

In [ ]:
z_rows = []
for _, row in labels.iterrows():
    pid = str(row["patient_id"])
    meta_path = processed_dir / pid / "crop_metadata.json"
    if not meta_path.exists():
        continue
    meta = json.loads(meta_path.read_text())
    z_rows.append({"patient_id": pid, **meta.get("zscore", {})})
z_df = pd.DataFrame(z_rows)
if not z_df.empty:
    print(z_df.describe())
z_df

## 2. Patient-level cross-validation (radiomics baseline)

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from src.features.baseline_classifier import RadiomicsBaseline

filtered = pd.read_csv(ROOT / "data" / "varfs_filtered_features.csv")
feature_cols = [c for c in filtered.columns if c not in ("patient_id", "label")]
x = filtered[feature_cols].fillna(0.0)
y = filtered["label"].astype(int)
n_folds = max(2, min(5, int(np.bincount(y).min() if len(y) else 2)))
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=int(cfg.get("seed", 42)))
fold_metrics = []
for fold, (tr, va) in enumerate(skf.split(x, y), start=1):
    clf = RadiomicsBaseline(cfg["baseline"])
    clf.train(filtered.iloc[tr])
    probs = clf.predict_proba(filtered.iloc[va])
    preds = (probs >= 0.5).astype(int)
    cm = confusion_matrix(y.iloc[va], preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    fold_metrics.append({
        "fold": fold,
        "auc": float(roc_auc_score(y.iloc[va], probs)) if len(set(y.iloc[va])) > 1 else float("nan"),
        "pr_auc": float(average_precision_score(y.iloc[va], probs)),
        "sensitivity": float(tp / (tp + fn)) if (tp + fn) else float("nan"),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else float("nan"),
    })
pd.DataFrame(fold_metrics)

## 3. Ablation comparison (radiomics vs SwinViT vs fused)

In [ ]:
metrics_path = results_dir / "evaluation_metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    pd.DataFrame(metrics).T
else:
    print("Run scripts/run_evaluation.py first.")

## 4. Interpretability sanity check

For each saved heatmap, measure the fraction of attention mass that falls inside
the liver mask vs the background. Values close to 1.0 indicate the model is
focused on liver parenchyma.

In [ ]:
rows = []
for hp in sorted(attention_dir.glob("*_attention.nii.gz")):
    pid = hp.stem.replace("_heatmap.nii", "")
    mask_path = processed_dir / pid / "before_liver.nii.gz"
    if not mask_path.exists():
        continue
    mask = nib.load(str(mask_path)).get_fdata() > 0
    heat = nib.load(str(hp)).get_fdata()
    total = float(heat.sum())
    inside = float(heat[mask].sum())
    rows.append({
        "patient_id": pid,
        "liver_attention_ratio": (inside / total) if total > 0 else float("nan"),
    })
pd.DataFrame(rows)